# EOS Consolidation Pipeline (Y2) -- Refactored

Same pipeline, restructured for correctness. Headline changes:

1. **Reconciliation cells fixed** -- column sets are now built programmatically, so the
   Opportunity-2-twice / Opportunity-3-missing copy-paste bug cannot recur. Checks run for
   every month, payments and counts, with a per-learner diff helper.
2. **No more CSV re-read mid-pipeline** -- `combined` is cleaned once, exported once
   (after cleaning), and every output format derives from that single in-memory frame.
3. **No silent row loss** -- the Internship type backfill is restored, all summary
   groupbys use `dropna=False`, and every drop prints what it dropped.
4. **`result` stays float** until export; int truncation was hiding cents-level drift
   in reconciliation.
5. **One config cell** -- all paths, the reporting window, and the FY month order live
   at the top.

Domain rules preserved as-is: XA-Sept-25 dedup exemption, the XH-PMA-Apr-25 surgical
drop, hardcoded email fixes, cohort remap, stipend exclusion from `singles`.


## 0. Config

In [ ]:

import pandas as pd
import numpy as np
from connect import connect_to_database
from retrieve_ids import get_ids_by_fields

# =====================================================================
# The three lines marked <-- MONTHLY EDIT are the monthly ritual.
# Everything else derives from them or changes rarely.
# =====================================================================

# ---- Reporting window (Y2 runs April 2026 - March 2027; edit yearly) ----
REPORT_START = pd.Timestamp('2026-04-01')
TODAY = pd.Timestamp('now').normalize()

# First day of the month being reported
CURRENT_MONTH = pd.Timestamp('2026-06-01')            # <-- MONTHLY EDIT

# prev_combined covers everything up to the end of last month; live-source rows
# dated on/before this that prev already contains are treated as re-reads
PREV_COVERS_THROUGH = CURRENT_MONTH - pd.Timedelta(days=1)

# ---- Folder layout ----
# Expected structure:
#   {BASE}/Monthly IDC/<Month (Year)>/Source Datasets/   <- data owner submissions
#   {BASE}/Monthly IDC/<Month (Year)>/Sink Datasets/     <- this notebook's outputs
# NOTE: folder naming on disk is not uniform -- some months have a space before
# the parenthesis ('June (2026)'), some do not ('May(2026)'). Match what exists.

# ---- Source paths ----
BASE = '/Users/markngendo/Documents/Umuzi/Reporting'

MONTH_DIR      = f'{BASE}/Monthly IDC/June (2026)'    # <-- MONTHLY EDIT
PREV_MONTH_DIR = f'{BASE}/Monthly IDC/May(2026)'      # <-- MONTHLY EDIT
Y1_DIR         = f'{BASE}/Monthly IDC/March(2026)'    # final Y1 month: fixed for all of Y2

SRC  = f'{MONTH_DIR}/Source Datasets'
SINK = f'{MONTH_DIR}/Sink Datasets'

# ---- Input files ----
# Monthly filenames follow the owners' usual pattern:
#   '<Team> <Month> <Year> Earning Opportunity Template.<ext>'
# ...but expect drift ('Copy of ' prefixes, owner names in brackets, exported
# sheet suffixes). The path check in the next cell lists what is actually in
# Source Datasets whenever a name does not match.
PATHS = {
    # database exports (stable locations)
    'rich':          f'{BASE}/Improved IDC/Categories/Support/db_fields.csv',
    'rich_fields':   f'{BASE}/Improved IDC/Email Matching/emails.csv',

    # monthly submissions
    'natalie':       f'{SRC}/Y2 Earning Opportunity _Continuous Updates_ - _XPL Natalie Jorgensen_.xlsx',  # continuous workbook: same name all year
    'milestone':     f'{SRC}/Finance June 2026 Earning Opportunity Template.xlsx',
    'partnerships':  f'{SRC}/SA Partnerships June 2026 Earning Opportunity Template [Kholofelo Mashigo].xlsx',
    'launch_lab':    f'{SRC}/Launch Lab June 2026 Earning Opportunity Template.xlsx - earning_opportunities.csv',
    'interns':       f'{SRC}/Umuzi - External Placements Template - Sheet1.csv',   # currently unused (loader commented out)
    'sap':           f'{SRC}/Copy of SAP June 2026 Earning Opportunity Template.xlsx - earning_opportunities.csv',
    'begreen':       f'{SRC}/BeGreen June 2026 Earning Opportunity Template [Mbali Zamisa].xlsx',

    # prior outputs feeding this run
    'prev_combined': f'{PREV_MONTH_DIR}/Sink Datasets/combined_eos.csv',  # last month's export of THIS notebook
    'y1_combined':   f'{Y1_DIR}/Sink Datasets/combined_eos.csv',          # Y1 final cumulative export (Counted Y1 baseline)

    # outputs
    'combined_out':  f'{SINK}/combined_eos.csv',
    'report_out':    f'{SINK}/IDC Report Consolidated Y2.xlsx',
}

# ---- Financial year month order (April-first), used everywhere ----
FY_MONTH_ORDER = {
    'April': 1, 'May': 2, 'June': 3, 'July': 4,
    'August': 5, 'September': 6, 'October': 7, 'November': 8,
    'December': 9, 'January': 10, 'February': 11, 'March': 12,
}

# ---- Known email typo corrections (applies to every source) ----
EMAIL_FIXES = {
    # Left empty for now since emails can't exist on a public repository
}

CANON_COLS = ['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
              'earning_opportunity_extra_info', 'earning_opportunity_type',
              'earning_opportunity_name']




In [ ]:
from pathlib import Path

Path(SINK).mkdir(parents=True, exist_ok=True)   # first run of a new month

SKIP = {'combined_out', 'report_out', 'interns'}   # outputs + unused source
missing = {k: p for k, p in PATHS.items() if k not in SKIP and not Path(p).exists()}

if missing:
    print('Missing input file(s):')
    for k, p in missing.items():
        print(f'  {k:14s} {p}')
    if Path(SRC).exists():
        print(f'\nActually in {SRC}:')
        for f in sorted(Path(SRC).iterdir()):
            print(f'  {f.name}')
    else:
        print(f'\n{SRC} does not exist -- check MONTH_DIR (folder naming varies).')
    raise FileNotFoundError(
        f'{len(missing)} CONFIG path(s) not found. Fix the filenames above '
        'using the directory listing, then re-run.')

if not Path(PATHS['report_out']).exists():
    raise FileNotFoundError(
        f"{PATHS['report_out']} does not exist yet. This notebook APPENDS to the "
        "workbook created by indicators_consolidated.ipynb -- run order is: "
        "support_services_consolidated -> indicators_consolidated -> this.")

print('All input paths exist; workbook ready for append.')

## 0.1 Helpers

In [2]:
def parse_date(val):
    """Parse a single date value across the formats the source sheets use."""
    formats = [
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y %H:%M:%S.%f",
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y",
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except Exception:
            continue
    try:
        return pd.to_datetime(val)
    except Exception:
        return pd.NaT


def clean_payment(series, source_name=''):
    """Strip currency formatting (R, spaces, commas) and coerce to numeric.
    Prints whatever fails to parse so nothing is dropped invisibly."""
    stripped = series.astype(str).str.replace(r'[R,\s]', '', regex=True)
    out = pd.to_numeric(stripped, errors='coerce')
    bad = series[out.isna() & series.notna()]
    if not bad.empty:
        print(f"[{source_name}] {len(bad)} unparseable payment value(s) -> dropped:")
        print(bad.value_counts().to_string())
    return out


def normalize_emails(series):
    """strip + lower + NFKC + typo fixes + personal-to-umuzi remap. One path for every source."""
    s = series.str.strip().str.lower().str.normalize('NFKC')
    s = s.replace(EMAIL_FIXES)
    return s.map(lambda x: emailMap.get(x, x))


def report_unmatched(series, source_name):
    """Emails in a source that have no enrichment record -- demographic gaps downstream."""
    unmatched = set(series.dropna()) - set(rich['umuzi_email'])
    if unmatched:
        print(f"[{source_name}] {len(unmatched)} email(s) not in enrichment data:")
        for e in sorted(unmatched):
            print(f"  {e}")
    return unmatched


## 1. Enrichment data

In [3]:
rich = pd.read_csv(PATHS['rich'], dtype={'id_number': str, 'cellphone_number': str})
rich['umuzi_email'] = rich['umuzi_email'].str.strip().str.lower()

richFields = pd.read_csv(PATHS['rich_fields'], dtype={'cellphone_number': str, 'id_number': str})
richFields['email'] = richFields['email'].str.strip().str.lower()

# personal email -> umuzi email map
emailMap = rich.set_index('email')['umuzi_email'].str.lower()


In [4]:
# ---- Y1 baseline: learners already counted in Y1 ----
y1 = pd.read_csv(PATHS['y1_combined'], usecols=['umuzi_email'])
y1_learners = set(normalize_emails(y1['umuzi_email'].dropna()))
print(f"{len(y1_learners)} unique Y1 learner(s) loaded as the Counted-Y1 baseline")

337 unique Y1 learner(s) loaded as the Counted-Y1 baseline


## 2. Source: Natalie (wide multi-opportunity sheets)

Three sheets in one workbook, each wide-format with repeating
`(payment, date, extra_info, type, name)` blocks per opportunity. Renamed positionally,
then melted to long.

In [5]:
def generate_column_rename_map(total_columns: int) -> dict:
    rename_map = {
        0: 'umuzi_email',
        1: 'cohort',
        2: 'payment_1',
        3: 'date_service_accessed_1',
        4: 'earning_opportunity_extra_info_1',
        5: 'earning_opportunity_type_1',
        6: 'earning_opportunity_name_1',
    }
    col_idx, opportunity_num = 7, 2
    while col_idx < total_columns:
        rename_map[col_idx]     = f'payment_{opportunity_num}'
        rename_map[col_idx + 1] = f'date_service_accessed_{opportunity_num}'
        rename_map[col_idx + 2] = f'earning_opportunity_extra_info_{opportunity_num}'
        rename_map[col_idx + 3] = f'earning_opportunity_type_{opportunity_num}'
        rename_map[col_idx + 4] = f'earning_opportunity_name_{opportunity_num}'
        col_idx += 5
        opportunity_num += 1
    return rename_map


def melt_opportunities(df):
    """Wide repeating-block layout -> one row per (learner, opportunity)."""
    return (
        pd.wide_to_long(
            df=df,
            stubnames=['earning_opportunity_type', 'earning_opportunity_name',
                       'payment', 'date_service_accessed', 'earning_opportunity_extra_info'],
            i=['umuzi_email', 'cohort'],
            j='payment_number',
            sep='_',
            suffix='\\d+',
        )
        .reset_index()
        .dropna(subset=['payment'])
        .drop(columns='payment_number')
    )


In [6]:
NATALIE_SHEETS = {0: range(1, 18), 1: range(1, 23), 2: range(1, 13), 3: range(1, 8)}

natalie_parts = []
for sheet_idx, usecols in NATALIE_SHEETS.items():
    df = pd.read_excel(io=PATHS['natalie'], sheet_name=sheet_idx, skiprows=1, usecols=usecols)

    mapper = generate_column_rename_map(df.shape[1])
    cols = df.columns.to_list()
    for idx, name in mapper.items():
        cols[idx] = name
    df.columns = cols
    df = df[list(mapper.values())]

    date_cols = [c for c in df.columns if c.startswith('date')]
    df[date_cols] = df[date_cols].map(parse_date)

    df = df.dropna(subset=['umuzi_email'])
    natalie_parts.append(melt_opportunities(df))

natalie = pd.concat(natalie_parts, ignore_index=True)
natalie['payment'] = clean_payment(natalie['payment'], 'natalie')
natalie = natalie.dropna(subset=['payment'])
natalie.shape


(289, 7)

## 3. Source: Milestone (Finance team)

Identified by SA ID number; resolved to umuzi_email via the enrichment table.
Rows whose identifier does not resolve are printed before being dropped --
the source column allows email **or** ID, so an email in that column will not
match and will show up below.

In [7]:
milestone = pd.read_excel(
    PATHS['milestone'], sheet_name=0, skiprows=1,
    dtype={'Umuzi Email Address OR ID #': 'str'},
    usecols=range(1, 8),
)

milestone = milestone.rename(columns={
    'Umuzi Email Address OR ID #': 'id_number',
    'Rand amount paid in a given month (can be zero for some gigs if learner on a stipend)': 'payment',
    "format DD/MM/YYYY - if only month and year known, put DD as 01 (if no date hasn't happened)": 'date_service_accessed',
    'If applicable - This is Useful for person compiing the data': 'cohort',
    'Useful for person compiing the data': 'earning_opportunity_extra_info',
    'type of earning opportunity accessed': 'earning_opportunity_name',
    'if needed to differentiate within a type (we can add to this list if need be - just need alignment between parties)': 'earning_opportunity_type',
})

milestone = milestone.dropna(subset=['id_number', 'date_service_accessed'])

milestone = milestone.merge(rich[['umuzi_email', 'id_number']], on='id_number', how='left')
unresolved = milestone[milestone['umuzi_email'].isna()]
if not unresolved.empty:
    print(f"[milestone] {len(unresolved)} row(s) with unresolvable identifier -> dropped:")
    print(unresolved['id_number'].to_string())

milestone = milestone.dropna(subset=['umuzi_email']).drop(columns='id_number')
milestone['date_service_accessed'] = milestone['date_service_accessed'].map(parse_date)
milestone['payment'] = clean_payment(milestone['payment'], 'milestone')
milestone.shape


(10, 7)

## 4. Source: Partnerships (6 sheets)

In [8]:
POSITIONAL_RENAME = {
    0: 'umuzi_email', 1: 'payment', 2: 'date_service_accessed', 3: 'cohort',
    4: 'earning_opportunity_extra_info', 5: 'earning_opportunity_type',
    6: 'earning_opportunity_name',
}

partnership_parts = []
for sht_idx in range(0, 5):
    df = pd.read_excel(io=PATHS['partnerships'], sheet_name=sht_idx, skiprows=1, usecols=range(1, 8))
    partnership_parts.append(df)

partnerships = pd.concat(partnership_parts, ignore_index=True)

cols = partnerships.columns.to_list()
for idx, name in POSITIONAL_RENAME.items():
    cols[idx] = name
partnerships.columns = cols

string_cols = partnerships.select_dtypes(include='object').columns
partnerships[string_cols] = partnerships[string_cols].apply(lambda x: x.str.strip())

partnerships['date_service_accessed'] = partnerships['date_service_accessed'].map(parse_date)
partnerships = partnerships.dropna(subset=['date_service_accessed', 'payment'])

# 'Done' / 'Learner...' placeholder strings coerce to NaN here and are printed+dropped
partnerships['payment'] = clean_payment(partnerships['payment'], 'partnerships')
partnerships = partnerships.dropna(subset=['payment'])

partnerships = partnerships.dropna(
    subset=['earning_opportunity_type', 'earning_opportunity_name'], how='all')
partnerships.shape


(49, 7)

## 5. Source: Launch Lab

In [9]:
llab = pd.read_csv(
    PATHS['launch_lab'],
    usecols=[1, 4, 7, 6, 5, 3, 2], skiprows=2,
    names=['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
           'earning_opportunity_extra_info', 'earning_opportunity_name', 'earning_opportunity_type'],

)

llab['payment'] = clean_payment(llab['payment'], 'launch_lab')
llab['date_service_accessed'] = llab['date_service_accessed'].map(parse_date)
llab.shape


(11, 7)

## 6. Source: Umuzi Interns

In [10]:
# interns = pd.read_csv(
#     PATHS['interns'],
#     usecols=[1, 3, 4, 5, 6, 7, 9], skiprows=1,
#     names=['umuzi_email', 'cohort', 'earning_opportunity_type', 'earning_opportunity_name',
#            'earning_opportunity_extra_info', 'date_service_accessed', 'payment'],
# )

# interns['payment'] = clean_payment(interns['payment'], 'interns')
# interns['date_service_accessed'] = interns['date_service_accessed'].map(parse_date)
# interns.shape


## 6.a Source: SAP

In [11]:
sap = pd.read_csv(
    PATHS['sap'],
    usecols=[1, 2, 3, 4, 5, 6, 7], skiprows=2,
    names=['umuzi_email', 'payment', 'date_service_accessed', 'cohort', 'earning_opportunity_extra_info',
           'earning_opportunity_name', 'earning_opportunity_type']
)
sap['payment'] = clean_payment(sap['payment'], 'sap')
sap['date_service_accessed'] = sap['date_service_accessed'].map(parse_date)
sap.shape
sap

,umuzi_email,payment,date_service_accessed,cohort,earning_opportunity_extra_info,earning_opportunity_name,earning_opportunity_type
0,marumo.mampolokeng@gmail.com,400,2026-06-24,SAP Educate to Employ Year 1,Impact gig (Freelance opportunity) payment for...,Freelance gig,Experience gig
1,bhekumusa.ntshwenya@umuzi.org,400,2026-06-24,SAP Educate to Employ Year 2,Impact gig (Freelance opportunity) payment for...,Freelance gig,Experience gig


## 6.b. Source: BeGreen

In [12]:
bg = pd.read_excel(
    PATHS['begreen'], dtype={'Umuzi Email Address OR ID #': 'str'},
    usecols=range(1, 8), skiprows=1
)
bg.columns = ['id_number', 'payment', 'date_service_accessed', 'cohort', 'earning_opportunity_extra_info', 'earning_opportunity_name', 'earning_opportunity_type']

bg = bg.dropna(subset=['id_number', 'date_service_accessed'])

bg= bg.merge(rich[['umuzi_email', 'id_number']], on='id_number', how='left')

unresolved = bg[bg['umuzi_email'].isna()]
if not unresolved.empty:
    print(f"[bg] {len(unresolved)} row(s) with unresolvable identifier -> dropped:")
    print(unresolved['id_number'].to_string())

bg = bg.dropna(subset=['umuzi_email']).drop(columns='id_number')
bg['date_service_accessed'] = bg['date_service_accessed'].map(parse_date)
bg['payment'] = clean_payment(bg['payment'], 'bg')

bg.shape

(26, 7)

## Source: Previous Combined

In [13]:
prev = pd.read_csv(PATHS['prev_combined'], usecols=CANON_COLS)
prev['date_service_accessed'] = prev['date_service_accessed'].map(parse_date)
prev.shape

(398, 7)

## 7. Combine and clean (single canonical pass)

Everything below derives from this one frame. The old flow exported `combined` to CSV
*before* several cleaning steps, then re-read that CSV later -- which silently restored
zero payments, subset duplicates, NaN dates, and unmapped emails into `result` and
`breakdown`. The export now happens once, after cleaning, and there is no re-read.

In [14]:
combined = pd.concat(
    [prev[CANON_COLS], natalie[CANON_COLS], milestone[CANON_COLS],
     partnerships[CANON_COLS], llab[CANON_COLS], sap[CANON_COLS], bg[CANON_COLS]],
    keys=['prev_combined', 'natalie', 'milestone', 'partnerships', 'launch_lab', 'sap', 'bg'],
    names=['source', None],
).reset_index(level='source').reset_index(drop=True)

# whitespace cleanup on every string column, once
STRING_COLS = ['umuzi_email', 'cohort', 'earning_opportunity_extra_info',
               'earning_opportunity_type', 'earning_opportunity_name']
combined[STRING_COLS] = combined[STRING_COLS].apply(lambda s: s.str.strip())

# one email normalization path for every source (strip/lower/NFKC + fixes + remap)
combined['umuzi_email'] = normalize_emails(combined['umuzi_email'])
report_unmatched(combined['umuzi_email'], 'combined')

combined['date_service_accessed'] = combined['date_service_accessed'].map(parse_date)
combined.shape


(785, 8)

In [15]:
# ---- Drop rows outside the reporting window or without a usable date ----
no_date = combined['date_service_accessed'].isna()
if no_date.any():
    print(f"{no_date.sum()} row(s) without a parseable date -> dropped")
    print(combined.loc[no_date, ['source', 'umuzi_email', 'payment']].to_string())
combined = combined[~no_date]

# NOTE: the old filter used `<= REPORT_START`, which excluded April 1 itself
# while the per-source asserts required >= April 1. Window is now inclusive.
out_of_window = (
    (combined['date_service_accessed'] < REPORT_START) |
    (combined['date_service_accessed'] > pd.Timestamp('now'))
)
if out_of_window.any():
    print(f"{out_of_window.sum()} row(s) outside reporting window -> dropped")
    display(combined[out_of_window])
combined = combined[~out_of_window]

combined['month'] = combined['date_service_accessed'].dt.month_name()


2 row(s) outside reporting window -> dropped


,source,umuzi_email,payment,date_service_accessed,cohort,earning_opportunity_extra_info,earning_opportunity_type,earning_opportunity_name
750,launch_lab,ndiyakholwa.mnqanqeni@umuzi.org,NaN,2026-03-01,C48\tNaspers,Developer - Full Stack,Placement,Employment
751,launch_lab,knoxenis@gmail.com,NaN,2025-08-01,C48\tNaspers,Project manager - Excelerate,Placement,Employment


In [16]:
# ---- Backfill missing earning_opportunity_type ----
# Internship rows were previously left NaN (the fill was commented out), which made
# them vanish from the breakdown groupby while remaining in result -- a direct
# reconciliation leak.
is_na_type = combined['earning_opportunity_type'].isna()
combined.loc[is_na_type & (combined['earning_opportunity_name'] == 'Internship'),
             'earning_opportunity_type'] = 'Placement'
combined.loc[is_na_type & (combined['earning_opportunity_name'] != 'Internship'),
             'earning_opportunity_type'] = 'Experience gig'

assert combined['earning_opportunity_type'].notna().all()


In [17]:
# ---- Deduplication (domain rules preserved) ----

# 1. exact duplicates
exact_dupes = combined.duplicated()
if exact_dupes.any():
    print(f"{exact_dupes.sum()} exact duplicate row(s) -> dropped")
combined = combined[~exact_dupes]

# 2. surgical drop: XH-PMA-Apr-25 rows on 2025-10-31 with no type
#    (kept from original -- note this date predates the Y2 window, so it should
#    match nothing now; retained in case the window changes)
surgical = (
    (combined['cohort'] == 'XH-PMA-Apr-25') &
    (combined['date_service_accessed'] == pd.Timestamp('2025-10-31')) &
    (combined['earning_opportunity_type'].isna())
)
if surgical.any():
    print(f"{surgical.sum()} XH-PMA-Apr-25 known-bad row(s) -> dropped")
combined = combined[~surgical]

# 3. subset duplicates caused by data owners re-submitting rows in later months.
#    XA-Sept-25 is exempt: identical-looking rows there are genuine repeat payments.
subset_dupe = combined.duplicated(
    subset=['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
            'earning_opportunity_extra_info', 'earning_opportunity_name'])
not_exempt = combined['cohort'] != 'XA-Sept-25'
drop_mask = subset_dupe & not_exempt
if drop_mask.any():
    print(f"{drop_mask.sum()} resubmitted duplicate row(s) (non-XA-Sept-25) -> dropped")
combined = combined[~drop_mask]

# 4. zero payments are excluded from monetary reporting (stipend gigs can be zero)
zero_pay = combined['payment'] == 0
if zero_pay.any():
    print(f"{zero_pay.sum()} zero-payment row(s) -> dropped")
combined = combined[~zero_pay]

nan_pay = combined['payment'].isna()
if nan_pay.any():
    print(f"{nan_pay.sum()} NaN-payment row(s) -> dropped")
    print(combined.loc[nan_pay, ['source', 'umuzi_email', 'month', 'date_service_accessed']].to_string())
combined = combined[~nan_pay]

combined = combined.reset_index(drop=True)
combined.shape


4 NaN-payment row(s) -> dropped
         source                   umuzi_email  month date_service_accessed
746  launch_lab    atlegang.sethono@umuzi.org  April            2026-04-01
752  launch_lab  siyabonga.mthethwa@umuzi.org   June            2026-06-17
755  launch_lab      wandisile.kupa@umuzi.org   June            2026-06-01
756  launch_lab    nkosinathi.nduli@umuzi.org   June            2026-06-01


(779, 9)

## 7.1 Cohort remap

In [18]:
remapCohort = {
    'XPL2': 'XA-Jun-25',
    'XPL5': 'XA-Nov-25',
    'XPL3': 'XA-Aug-25',
    'XPL4': 'XA-Sept-25',
    'XPL7': 'XA-Jun-26',
    'XA-Apr-26': 'XA-Apr-26',
    'C44 - BBD Learnership': 'c44_wd',
    'XPL - Project Management': 'XH-PMA-Apr-25',
    'XZ1 (Foundational Web Dev)': 'c46_wd',
    'XZ1 ( Foundational Web Dev)': 'c46_wd',
    'C19 - Grey Learnership': 'c19_cl',
    'C45 (Naspers Skills Programme)': 'c45_de',
    'VML_UI/UX': 'XH-AUXUI-Aug-25',
    'XA1_AWD_2025': 'c44_wd',
    'XM1_PM_2025': 'XH-PMA-Apr-25',
    'XH-PM-Apr-25': 'XH-PMA-Apr-25',
    'XSD1_2025': 'XH-SD-Jun-25',
    'XZ1 (AWD)': 'a4_rn',
    'A3 0 BBD': 'a3_rn',
    'A3 BBD': 'a3_rn',
    'A4 BBD': 'a4_rn',
    'A5 - BBD': 'XH-AWD-BBD-Feb-26',
    'C45 - Naspers 2024': 'c45_de',
    'C45': 'c45_ux_s',
    'C39': 'c39_java',
    'EE SA_Adv_Web_Dev': 'XH-AWD-Aug-25',
    'SL-PM-WES-INT-Nov25': 'SL-PM-Wesbank-Nov-25',
    'XPL-Hybrid-Salesforce-Developer': 'XH-SD-Jun-25',
    'XPL Hybrid Salesforce Dev Jun2025 - Project Y': 'XH-SD-Jun-25',
    'XPL Web Dev Abrig Apr2025 - Project Y': 'XH-AWDA-Apr-25',
    'Project Y': 'XH-AWDA-Apr-25',
    'XPL Hybrid Apr25 - Naspers': 'XH-AWD-Apr-25',
    'XPL Accelerator - Sept 2025': 'XA-Sept-25',
    'XPL Accelerator - Aug 2025': 'XA-Aug-25',
    'Accelerator bootcamp': 'XA-Sept-25',
    'C48 Naspers': 'XH-AWD-Apr-25',
    'C45 Naspers': 'XA-Sept-25',
    'C16 Investec': 'c16_wd2',
    'C35 Sanlam': 'c35_ds1',
    'C46 - BBD': 'c46_wd',
    'C46 - BDD': 'c46_wd',
    'C41': 'c41_ds',
    'SAP Educate to Employ': 'SAP Educate to Employ',
    'SAP Educate to Employ Year 1': 'SAP Educate to Employ',
    'SAP Educate to Employ Year 2': 'SAP Educate to Employ',
    'SAP Y1': 'SAP Y1',
    'SAP Y2': 'SAP Y2',
    'SL-FWD-Feb26': 'SL-FWD-Feb26',
    'XA-Aug-25': 'XA-Aug-25',
    'XA-Jun-25': 'XA-Jun-25',
    'XA-Nov-25': 'XA-Nov-25',
    'XA-Sept-25': 'XA-Sept-25',
    'XH-AUXUI-Aug-25': 'XH-AUXUI-Aug-25',
    'XH-AWD-Apr25': 'XH-AWD-Apr-25',
    'XH-AWD-Aug-25': 'XH-AWD-Aug-25',
    'XH-AWD-Feb26': 'XH-AWD-BBD-Feb-26',
    'XH-SD-Jun-25': 'XH-SD-Jun-25',
    'a3_rn': 'a3_rn',
    'a4_rn': 'a4_rn',
    'c19_cl': 'c19_cl',
    'c39_java': 'c39_java',
    'c44_wd': 'c44_wd',
    'c45_de': 'c45_de',
    'c45_ux_s': 'c45_ux_s',
    'c46_wd': 'c46_wd',
    'c16_wd2': 'c16_wd2',
    'c35_ds1': 'c35_ds1',
    'c41_ds': 'c41_ds',
    'SL-PM-Wesbank-Nov-25': 'SL-PM-Wesbank-Nov-25',
    'XH-AWD-Apr-25': 'XH-AWD-Apr-25',
    'XH-AWD-BBD-Feb-26': 'XH-AWD-BBD-Feb-26',
    'XH-AWDA-Apr-25': 'XH-AWDA-Apr-25',
    'XH-PMA-Apr-25': 'XH-PMA-Apr-25',
    'XPL6': 'XA-Apr-26',
    'XPL Accelerator - November 2025': 'XA-Nov-25',
    'WesBank Internship': 'SL-PM-Wesbank-Nov-25',
    'Wesbank Intern': 'XH-AWDA-Apr-25',
    'BG': 'BeGreen Cohort',
    'C41\tEqual Experts': 'SL Data Science Learnership - November 2023 (C41)',
    'C49\tEqual Experts': 'SL Web Development Learnership - March 2024 (C45)',
    'C50\tVMLYR': 'XPL Hybrid Advanced UX/UI - August 2025',
}

# Surface unmapped cohorts BEFORE mapping. Previously .map() turned any unmapped
# cohort into NaN, which then fell into 'Missing Cohort Info' downstream.
unmapped = set(combined['cohort'].dropna()) - set(remapCohort)
if unmapped:
    print(f"WARNING: {len(unmapped)} cohort value(s) not in remapCohort (will become NaN):")
    for u in sorted(unmapped):
        print(f"  {u!r}")

combined['cohort'] = combined['cohort'].map(remapCohort)


  'Wesbank_Intern'


In [19]:
# re-run resubmission dedup now that cohorts are canonical -- catches
# May-export rows colliding with June source rows across the remap boundary
post_remap_dupe = combined.duplicated(
    subset=['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
            'earning_opportunity_extra_info', 'earning_opportunity_name'])
drop_mask = post_remap_dupe & (combined['cohort'] != 'XA-Sept-25')
if drop_mask.any():
    print(f"{drop_mask.sum()} duplicate row(s) surfaced after cohort remap -> dropped")
    print(combined.loc[drop_mask, ['source', 'umuzi_email', 'month', 'payment']].to_string())
combined = combined[~drop_mask].reset_index(drop=True)

145 duplicate row(s) surfaced after cohort remap -> dropped
      source                       umuzi_email  month  payment
486  natalie     abongile.gangathele@umuzi.org  April   3000.0
487  natalie     abongile.gangathele@umuzi.org    May   2250.0
488  natalie     abongile.gangathele@umuzi.org    May   1250.0
489  natalie     abongile.gangathele@umuzi.org    May   5000.0
490  natalie            gambu.memela@umuzi.org  April   3000.0
491  natalie            gambu.memela@umuzi.org    May   2250.0
492  natalie            gambu.memela@umuzi.org    May   2250.0
493  natalie            gambu.memela@umuzi.org    May   5000.0
494  natalie       itumeleng.monaune@umuzi.org  April   3000.0
495  natalie       itumeleng.monaune@umuzi.org    May   2250.0
496  natalie       itumeleng.monaune@umuzi.org    May   2250.0
497  natalie       itumeleng.monaune@umuzi.org    May   5000.0
498  natalie      lindokuhle.manciya@umuzi.org  April   3000.0
499  natalie      lindokuhle.manciya@umuzi.org    May   12

In [20]:
KEY = ['umuzi_email', 'payment', 'date_service_accessed', 'cohort',
       'earning_opportunity_extra_info', 'earning_opportunity_name']

prev_counts = (combined.loc[combined['source'] == 'prev_combined']
               .groupby(KEY, dropna=False).size().rename('in_prev').reset_index())

old_live = ((combined['source'] != 'prev_combined') &
            (combined['date_service_accessed'] <= PREV_COVERS_THROUGH))

ol = combined.loc[old_live].reset_index()          # keep original index for the drop
ol['rank'] = ol.groupby(KEY, dropna=False).cumcount()
ol = ol.merge(prev_counts, on=KEY, how='left')
resub = ol['rank'] < ol['in_prev'].fillna(0)

print(f"{resub.sum()} pre-June live row(s) already covered by prev_combined -> dropped")
backfills = ol.loc[~resub]
if not backfills.empty:
    print(f"{len(backfills)} genuine back-fill row(s) kept -- verify with data owners:")
    print(backfills.groupby(['source', 'cohort'], dropna=False).size().to_string())

combined = combined.drop(index=ol.loc[resub, 'index']).reset_index(drop=True)

88 pre-June live row(s) already covered by prev_combined -> dropped
1 genuine back-fill row(s) kept -- verify with data owners:
source      cohort                                 
launch_lab  XPL Hybrid Advanced UX/UI - August 2025    1


In [21]:
# ---- Window asserts + single export of the CLEAN frame ----
assert combined['date_service_accessed'].min() >= REPORT_START, "Pre-window date survived cleaning"
assert combined['date_service_accessed'].max() <= pd.Timestamp('now'), "Future date survived cleaning"
assert not combined.duplicated().any()

combined.drop(columns='source').to_csv(PATHS['combined_out'], index=False)
combined.shape


(546, 9)

## 8. Programme names (DB slug map + manual fallback)

In [22]:
conn = connect_to_database()

slugMapRows = get_ids_by_fields(
    fields={'slug', 'name'},
    table='programmes',
    conn=conn,
    filters={'slug': set(combined['cohort'].fillna('Missing Value'))},  # type: ignore
)
slugMap = {entry['slug']: entry['name'] for entry in slugMapRows}

# manual fallback for slugs the DB does not return
slug_to_name = {
    'XH-SD-Jun-25': 'XPL Hybrid Salesforce Developer - June 2025',
    'XH-PMA-Apr-25': 'XPL Hybrid Project Management Abridged - April 2025',
    'XH-AWD-BBD-Feb-26': 'XPL Hybrid Advanced Web Development - BBD - February 2026 (A5)',
    'XH-AWD-Aug-25': 'XPL Hybrid Advanced Web Development - August 2025',
    'XH-AWD-Apr-25': 'XPL Hybrid Advanced Web Development - April 2025',
    'XH-AWDA-Apr-25': 'XPL Hybrid Advanced Web Development Abridged - April 2025',
    'XH-AUXUI-Aug-25': 'XPL Hybrid Advanced UX/UI - August 2025',
    'XA-Sept-25': 'XPL Accelerator - September 2025',
    'XA-Nov-25': 'XPL Accelerator - November 2025',
    'XA-Jun-25': 'XPL Accelerator - June 2025',
    'XA-Aug-25': 'XPL Accelerator - August 2025',
    'SL-PM-Wesbank-Nov-25': 'SL Project Management Wesbank Internship',
    'SL-FWD-Feb26': 'SL Foundational Web Development - February 2026',
    'c46_wd': 'SL Web Development Learnership - February 2025 (C46)',
    'c45_ux_s': 'SL UX Strategy Learnership - March 2024 (C45)',
    'c45_de': 'SL Data Engineering Learnership - March 2024 (C45)',
    'c44_wd': 'SL Web Development Learnership - April 2024 (C44)',
    'c41_ds': 'SL Data Science Learnership - November 2023 (C41)',
    'c39_java': 'SL Java Learnership - June 2023 (C39)',
    'c35_ds1': 'SL Data Science Learnership - May 2022 (C35)',
    'c19_cl': 'SL Copywriting Learnership - December 2019 (C19)',
    'c16_wd2': 'SL Web Development Learnership - September 2019 (C16)',
    'a4_rn': 'SL React Native Advanced - February 2025 (A4)',
    'a3_rn': 'SL React Native Advanced - April 2024 (A3)',
}

# DB wins where present; manual map fills the gaps
programmeMap = {**slug_to_name, **slugMap}


2026-07-09 08:44:48,687 INFO applications inserts logger: Connection to specified postgres is successful!


## 9. Demographics enrichment

Three-tier match (rich by umuzi_email, then richFields by personal email, then
richFields by umuzi_email), exactly as before. The dedup key is the same; the only
change is that this now runs against the *cleaned* `combined`.

In [23]:
neededCols = ['umuzi_email', 'cellphone_number', 'id_number', 'gender', 'race',
              'residential_area_type', 'province', 'nearest_metro',
              'has_disability_or_differently_abled', 'date_of_birth',
              'cohort', 'earning_opportunity_extra_info', 'earning_opportunity_type',
              'earning_opportunity_name', 'date_service_accessed',
              'first_name', 'last_name', 'payment', 'learner_id']

DEMO_FIELDS = ['cellphone_number', 'id_number', 'race', 'gender', 'date_of_birth',
               'nearest_metro', 'has_disability_or_differently_abled', 'province',
               'residential_area_type', 'first_name', 'last_name', 'learner_id']

rich_u = rich.sort_values('learner_id').drop_duplicates(subset='umuzi_email', keep='last')
richFields_email_u = richFields.sort_values('learner_id').drop_duplicates(subset='email', keep='last')
richFields_umuzi_u = richFields.sort_values('learner_id').drop_duplicates(subset='umuzi_email', keep='last')

assert rich_u['umuzi_email'].is_unique
assert richFields_email_u['email'].is_unique
assert richFields_umuzi_u['umuzi_email'].is_unique


In [24]:
first_merge = combined.merge(
    rich_u[['umuzi_email'] + DEMO_FIELDS], on='umuzi_email', how='left')

second_merge = combined.merge(
    richFields_email_u[['email'] + DEMO_FIELDS],
    left_on='umuzi_email', right_on='email', how='left')

third_merge = combined.merge(
    richFields_umuzi_u[['umuzi_email'] + DEMO_FIELDS], on='umuzi_email', how='left')

demographics = (
    first_merge
    .combine_first(second_merge)
    .combine_first(third_merge)
    .drop_duplicates(subset=['umuzi_email', 'earning_opportunity_extra_info',
                             'earning_opportunity_type', 'earning_opportunity_name',
                             'date_service_accessed', 'payment'], keep='first')
    [neededCols]
)

demographics['learner_id'] = demographics['learner_id'].astype('Int64')


In [25]:
# age + reporting month -- dates are already datetime, no re-parsing needed
dob = demographics['date_of_birth'].map(parse_date)

demographics['age_service_accessed'] = (
    (demographics['date_service_accessed'] - dob).dt.days // 365
)

demographics['age_range'] = pd.cut(
    x=demographics['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+'],
)

demographics['month_of_service_accessed'] = (
    demographics['date_service_accessed'].dt.month_name()
    + ' '
    + demographics['date_service_accessed'].dt.year.astype(str)
)

# learners we still cannot phone -- worth chasing with the data owners
no_phone = (
    demographics.loc[demographics['cellphone_number'].isna(), 'umuzi_email']
    .pipe(lambda s: s[s.str.contains('umuzi', na=False)])
    .unique()
)
print(f"{len(no_phone)} umuzi-email learner(s) without a phone number")
no_phone


0 umuzi-email learner(s) without a phone number


array([], dtype=object)

## 10. Wide format (Monthly Entries Breakdown)

In [26]:
combined['row_id'] = combined.groupby(by=['umuzi_email', 'month']).cumcount()

wide = combined.pivot(
    index=['umuzi_email', 'cohort', 'row_id'],
    columns='month',
    values=['payment', 'earning_opportunity_type',
            'earning_opportunity_name', 'earning_opportunity_extra_info'],
)
wide.columns = [f'{month}_{col}' for col, month in wide.columns]
wide = wide.reset_index().drop(columns='row_id')

months_present = sorted(
    {c.split('_')[0] for c in wide.columns if '_' in c and c.split('_')[0] in FY_MONTH_ORDER},
    key=lambda m: FY_MONTH_ORDER[m],
)
fields = ['payment', 'earning_opportunity_type',
          'earning_opportunity_name', 'earning_opportunity_extra_info']

wide = wide.reindex(columns=['umuzi_email', 'cohort']
                    + [f'{m}_{f}' for m in months_present for f in fields])
combined = combined.drop(columns='row_id')
wide.head()


,umuzi_email,cohort,April_payment,April_earning_opportunity_type,April_earning_opportunity_name,April_earning_opportunity_extra_info,May_payment,May_earning_opportunity_type,May_earning_opportunity_name,May_earning_opportunity_extra_info,June_payment,June_earning_opportunity_type,June_earning_opportunity_name,June_earning_opportunity_extra_info
0,1996eddiwardo@gmail.com,BeGreen Cohort,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40775.0,Development pathway,Seed Funding,Begreen First Tranch grant payment
1,aaron.motlhale@umuzi.org,XA-Jun-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,350.0,Development pathway,XPL Accelerator Pathway,Block 1 Outcome 1
2,abongile.gangathele@umuzi.org,XA-Nov-25,3000.0,Experience gig,Professional gig,Block 4 Gig 1,2250.0,Experience gig,Technical gig,Block 4 Gig 2,NaN,NaN,NaN,NaN
3,abongile.gangathele@umuzi.org,XA-Nov-25,NaN,NaN,NaN,NaN,1250.0,Experience gig,Social gig,Block 4 Gig 3,NaN,NaN,NaN,NaN
4,abongile.gangathele@umuzi.org,XA-Nov-25,NaN,NaN,NaN,NaN,5000.0,Experience gig,Professional gig,Block 5 Gig 1,NaN,NaN,NaN,NaN


## 11. Result format (Summarized Opportunities)

Stays **float** until export. The old `.astype(int)` truncated cents per learner-month,
which put a permanent small skew into every reconciliation against the float breakdown.

In [27]:
result = (
    combined
    .groupby(by=['umuzi_email', 'month'], as_index=False)
    .agg(payment_sum=('payment', 'sum'), payment_count=('payment', 'count'))
    .pivot(index='umuzi_email', columns='month', values=['payment_sum', 'payment_count'])
    .fillna(0)
    .reset_index()
)

result.columns = ['umuzi_email'] + [
    f'{month} (Payment)' if metric == 'payment_sum' else f'{month} (Count)'
    for metric, month in result.columns[1:]
]

ordered = ['umuzi_email'] + sorted(
    result.columns[1:], key=lambda c: FY_MONTH_ORDER[c.split()[0]])
result = result[ordered]
result.head()


,umuzi_email,April (Payment),April (Count),May (Payment),May (Count),June (Payment),June (Count)
0,1996eddiwardo@gmail.com,0.0,0.0,0.0,0.0,40775.0,1.0
1,aaron.motlhale@umuzi.org,0.0,0.0,0.0,0.0,350.0,1.0
2,abongile.gangathele@umuzi.org,3000.0,1.0,13500.0,4.0,0.0,0.0
3,adumu.abdullahi@umuzi.org,3500.0,1.0,3500.0,1.0,3500.0,1.0
4,akonaho.muneri@umuzi.org,0.0,0.0,0.0,0.0,350.0,1.0


## 12. Breakdown format (Earning Opportunities Secured)

Two structural fixes:
- every groupby/pivot uses `dropna=False`, so rows with a missing key can no longer
  silently vanish from this sheet while staying in `result`
- `umuzi_email` is **kept** on the frame until reconciliation passes; it is only
  dropped at export

In [28]:
eon_rank = (
    combined
    .groupby(['umuzi_email', 'earning_opportunity_type', 'earning_opportunity_name'],
             as_index=False, dropna=False)
    .agg(first_eon_date=('date_service_accessed', 'min'))
)
eon_rank['op_num'] = (
    eon_rank
    .sort_values('first_eon_date')
    .groupby(['umuzi_email', 'earning_opportunity_type'], dropna=False)
    .cumcount() + 1
)

pivotData = combined.merge(
    eon_rank,
    on=['umuzi_email', 'earning_opportunity_type', 'earning_opportunity_name'],
    how='left',
)
pivotData['op_num'] = pivotData['op_num'].astype('Int64')
pivotData['cohort'] = pivotData['cohort'].fillna('Missing Cohort Info')

# must be zero now that types are backfilled and keys are never dropped
missing_op = pivotData['op_num'].isna().sum()
print(f'rows with missing op_num: {missing_op}')
assert missing_op == 0, 'op_num failed to assign -- check eon_rank merge keys'


rows with missing op_num: 0


In [29]:
a = (
    pivotData
    .groupby(['umuzi_email', 'earning_opportunity_type', 'cohort', 'op_num', 'month'],
             as_index=False, dropna=False)
    .agg(amount=('payment', 'sum'), gigs=('payment', 'size'))
)

first_dates = (
    combined
    .groupby(['umuzi_email', 'earning_opportunity_type'], as_index=False, dropna=False)
    .agg(first_date=('date_service_accessed', 'min'))
)
a = a.merge(first_dates, on=['umuzi_email', 'earning_opportunity_type'], how='left')


index_cols = ['umuzi_email', 'first_date', 'earning_opportunity_type', 'cohort']
assert a[index_cols].notna().all().all(), 'NaN in pivot index keys -- fix upstream, do not use dropna=False'

pivotA = a.pivot_table(
    index=index_cols,
    columns=['month', 'op_num'],
    values=['amount', 'gigs'],
    fill_value=0,
    aggfunc='sum',
)
# the all-zero column trim line can be deleted -- dropna=True only
# creates observed column combos
# pivot_table(dropna=False) keeps all column combos incl. empty ones; trim all-zero cols
pivotA = pivotA.loc[:, (pivotA != 0).any(axis=0)]

pivotA = pivotA.sort_index(axis=1, level=[1, 2, 0])
pivotA.columns = [
    (f'Earning Opportunity {op_num} ZAR Value ({month})'
     if metric == 'amount' else f'# of Gig {op_num} ({month})')
    for metric, month, op_num in pivotA.columns
]
pivotA = pivotA.reset_index()


In [30]:
value_cols = [c for c in pivotA.columns if c.startswith('Earning Opportunity ')]
gig_cols = [c for c in pivotA.columns if c.startswith('# of')]
assert len(value_cols) == len(gig_cols), 'value/gig column mismatch -- check pivot'

def month_of(col):
    return col.split('(')[-1].replace(')', '').strip()

def opp_sort_key(col):
    return (FY_MONTH_ORDER[month_of(col)], int(col.split()[2]))

def gig_sort_key(col):
    return (FY_MONTH_ORDER[month_of(col)], int(col.split()[3]))

sorted_opportunities = sorted(value_cols, key=opp_sort_key)
sorted_gigs = sorted(gig_cols, key=gig_sort_key)


In [31]:
demo = demographics.assign(
    umuzi_email=demographics['umuzi_email'].str.strip().str.lower().str.normalize('NFKC')
).drop_duplicates(subset=['umuzi_email'])

breakdown = pivotA.merge(
    demo[['umuzi_email', 'id_number', 'cellphone_number', 'nearest_metro',
          'province', 'race', 'gender', 'first_name', 'last_name']],
    on='umuzi_email',
    how='left',
)

# fall back to email where the name is missing instead of asserting and stopping
breakdown['Full Name'] = (
    (breakdown['first_name'] + ' ' + breakdown['last_name'])
    .fillna(breakdown['umuzi_email'])
)
nameless = breakdown['first_name'].isna().sum()
if nameless:
    print(f'{nameless} record(s) without a name -- email used as Full Name')

breakdown = breakdown.drop(columns=['first_name', 'last_name'])

statusDict = {
    'Development pathway': 'Active',
    'Placement': 'Placement',
    'Placed by Umuzi': 'Placement',
    'Impact gig': 'Active',
    'Experience gig': 'Active',
    'Learnership': 'Active'
}
unmapped_status = set(breakdown['earning_opportunity_type'].dropna()) - set(statusDict)
if unmapped_status:
    print(f'WARNING: types without a status mapping: {unmapped_status}')

breakdown['Programme'] = breakdown['cohort'].map(lambda x: programmeMap.get(x, 'Not Provided'))
breakdown['Current Status'] = breakdown['earning_opportunity_type'].map(statusDict)

# Y2 continuity flag: True = ongoing from Y1, False = counted anew in Y2
breakdown['Counted Y1'] = breakdown['umuzi_email'].isin(y1_learners)

# first row per learner counts them; later rows are 'Counted'
is_first_row = breakdown.groupby('umuzi_email').cumcount() == 0
breakdown['Youth Count (1 vs Counted)'] = np.where(
    is_first_row & ~breakdown['Counted Y1'], '1', 'Counted')


# umuzi_email stays on the frame until reconciliation passes; dropped at export
columnOrder = (
    ['umuzi_email', 'Full Name', 'id_number', 'first_date', 'Youth Count (1 vs Counted)', 'Counted Y1', 'Current Status',
     'cellphone_number', 'nearest_metro', 'province', 'cohort', 'Programme',
     'race', 'gender', 'earning_opportunity_type']
    + [item for pair in zip(sorted_opportunities, sorted_gigs) for item in pair]
)
breakdown = breakdown[columnOrder]
breakdown.sample(min(10, len(breakdown)))


,umuzi_email,Full Name,id_number,first_date,Youth Count (1 vs Counted),Counted Y1,Current Status,cellphone_number,nearest_metro,province,...,Earning Opportunity 2 ZAR Value (May),# of Gig 2 (May),Earning Opportunity 3 ZAR Value (May),# of Gig 3 (May),Earning Opportunity 1 ZAR Value (June),# of Gig 1 (June),Earning Opportunity 2 ZAR Value (June),# of Gig 2 (June),Earning Opportunity 3 ZAR Value (June),# of Gig 3 (June)
65,junior.masete@umuzi.org,Junior Masete,0004205309083,2026-06-28,Counted,True,Active,+27677441100,Johannesburg,NaN,...,0.0,0,0.0,0,4000.0,1,0.0,0,0.0,0
71,kgomotso.mongale@umuzi.org,Kgomotso Mongale,0201315702088,2026-04-10,Counted,True,Active,+27670627051,Johannesburg,NaN,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
151,ndumiso.khoza@umuzi.org,Ndumiso Khoza,9608156215080,2026-06-22,Counted,True,Active,+27762027612,Nelspruit,NaN,...,0.0,0,0.0,0,6000.0,1,0.0,0,0.0,0
66,kamogelo.nabane@umuzi.org,kamogelo nabane,0505125872083,2026-05-06,1,False,Active,+27636342370,Pretoria,NaN,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
33,brian.malau@umuzi.org,Brian Mahlatse Malau,0010235559084,2026-04-10,Counted,True,Active,+27790883791,Johannesburg,NaN,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
84,lesego.mbenge@umuzi.org,Lesego Mbenge,9903156451086,2026-04-22,Counted,True,Active,+27679352476,Rustenburg,North West,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
41,ditlhare.foloko@umuzi.org,Ditlhare Foloko,9701080309085,2026-04-10,Counted,True,Active,+27625091559,Johannesburg,NaN,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
60,itumeleng.mpye@umuzi.org,Itumeleng Mpye,9803220323081,2026-06-09,1,False,Active,+27718386535,Johannesburg,NaN,...,0.0,0,0.0,0,350.0,1,0.0,0,0.0,0
100,lunga.ngwenya@umuzi.org,Lunga Ngwenya,0208165142083,2026-06-09,1,False,Active,+27799230967,Pretoria,Gauteng,...,0.0,0,0.0,0,350.0,1,0.0,0,0.0,0
184,olwethu.manqola@umuzi.org,Olwethu Manqola,0203031240088,2026-04-22,Counted,True,Active,+27608114930,Soweto,Gauteng,...,0.0,0,0.0,0,8000.0,1,0.0,0,0.0,0


In [32]:
n_new = breakdown.loc[~breakdown['Counted Y1'], 'umuzi_email'].nunique()
assert (breakdown['Youth Count (1 vs Counted)'] == '1').sum() == n_new, \
    "Youth Count '1' rows must equal unique non-Y1 learners -- check Counted Y1 assignment order"

n_ongoing = breakdown.loc[breakdown['Counted Y1'], 'umuzi_email'].nunique()
n_new = breakdown['umuzi_email'].nunique() - n_ongoing
print(f"Y2 learners: {n_new} counted anew, {n_ongoing} ongoing from Y1")

Y2 learners: 110 counted anew, 141 ongoing from Y1


## 13. Reconciliation

The old check hardcoded the opportunity columns -- and listed Opportunity 2 twice
while omitting Opportunity 3 entirely (in both the payment and count cells), so the
reported disparity was really `(Opp 3 + ...) - Opp 2`. Column sets are now built from
the data, every month is checked, and a per-learner diff pinpoints any leak.

In [33]:
def reconcile(result, breakdown, atol=0.01):
    """result vs breakdown, every month, payments and counts.
    positive diff = rows missing from breakdown; negative = extra rows in breakdown."""
    months = sorted(
        {c.split()[0] for c in result.columns if c.endswith('(Payment)')},
        key=lambda m: FY_MONTH_ORDER[m],
    )
    rows, ok = [], True
    for m in months:
        val_cols = [c for c in breakdown.columns
                    if c.startswith('Earning Opportunity') and c.endswith(f'({m})')]
        cnt_cols = [c for c in breakdown.columns
                    if c.startswith('# of Gig') and c.endswith(f'({m})')]

        pay_diff = result[f'{m} (Payment)'].sum() - breakdown[val_cols].sum().sum()
        cnt_diff = result[f'{m} (Count)'].sum() - breakdown[cnt_cols].sum().sum()
        rows.append({'month': m, 'n_value_cols': len(val_cols),
                     'payment_diff': round(pay_diff, 2), 'count_diff': cnt_diff})
        if abs(pay_diff) > atol or cnt_diff != 0:
            ok = False

    report = pd.DataFrame(rows)
    print(report.to_string(index=False))
    return ok, report


def per_learner_diff(result, breakdown, month):
    """Who exactly differs, and by how much, for one month."""
    val_cols = [c for c in breakdown.columns
                if c.startswith('Earning Opportunity') and c.endswith(f'({month})')]
    bd = breakdown.groupby('umuzi_email')[val_cols].sum().sum(axis=1)
    rs = result.set_index('umuzi_email')[f'{month} (Payment)']
    diff = rs.sub(bd, fill_value=0)
    return diff[diff.abs() > 0.005].sort_values()


In [34]:
recon_ok, recon_report = reconcile(result, breakdown)

if not recon_ok:
    worst = recon_report.loc[recon_report['payment_diff'].abs().idxmax(), 'month']
    print(f'\nPer-learner diff for {worst}:')
    print(per_learner_diff(result, breakdown, worst).to_string())

assert recon_ok, 'result and breakdown do not reconcile -- see diff above'


month  n_value_cols  payment_diff  count_diff
April             2           0.0         0.0
  May             3           0.0         0.0
 June             3           0.0         0.0


In [35]:
# # Run this in case there's a disparity in counts but not payment
# combined[combined['payment'].isna()].groupby(['source', 'month']).size()

## 14. Remaining sheets (Learner Demographics, One Per Learner)

In [36]:
demo_columns = ['umuzi_email', 'cellphone_number', 'id_number', 'gender', 'race',
                'residential_area_type', 'province', 'nearest_metro',
                'has_disability_or_differently_abled', 'date_of_birth']

learner_demographics = (
    result[['umuzi_email']]
    .merge(rich[demo_columns], on='umuzi_email', how='left')
)


In [37]:
columnsEops = ['umuzi_email', 'learner_id'] + [
    col for col in demographics.columns
    if col not in demo_columns + ['first_name', 'last_name', 'learner_id']
]
singles = demographics[columnsEops].copy()

# stipends are excluded from the one-per-learner sheet
stipend_mask = singles['earning_opportunity_name'].isin(
    ['Data Stipend', 'Data Stipend, Exam stipend'])
print(f'{stipend_mask.sum()} stipend row(s) excluded from singles')
singles = singles[~stipend_mask]


0 stipend row(s) excluded from singles


In [38]:
demographics.shape, wide.shape, result.shape, breakdown.shape, singles.shape, learner_demographics.shape


((546, 22), (408, 14), (251, 7), (296, 31), (546, 11), (251, 10))

In [39]:
assert breakdown['umuzi_email'].nunique() == result['umuzi_email'].nunique()
# breakdown['umuzi_email'].value_counts().pipe(lambda s: s[s > 1])  # the multi-row learners

## 15. Final exports

Rounding and the email-column drop happen **here**, after reconciliation has passed,
not upstream where they corrupt the checks.

In [40]:
PARTNER_RENAME = {
    'Full Name': 'Name & Surname',
    'id_number': 'SA ID Number',
    'first_date': 'Date First Earning Opportunity Secured',
    'cellphone_number': 'Cell Number',
    'nearest_metro': 'Geographic Location (City)',
    'province': 'Geographic Location (Province)',
    'cohort': 'Cohort',
    'Programme': 'Umuzi-PYEI programme (s) participating in',
    'race': 'Race',
    'gender': 'Gender',
    'earning_opportunity_type': 'Earning Opportunities Type',
}

def month_year(m):
    # FY runs April->March; Jan-Mar belong to the following calendar year
    year = REPORT_START.year if FY_MONTH_ORDER[m] <= 9 else REPORT_START.year + 1
    return f'{m} {year}'

def partner_col(col):
    # partner quirk: only Opportunity 1 of each month carries the year
    if col.startswith('Earning Opportunity ') and col.split()[2] == '1':
        m = col.split('(')[-1].rstrip(')')
        return col.replace(f'({m})', f'({month_year(m)})')
    return PARTNER_RENAME.get(col, col)

breakdown_export = breakdown.drop(columns='umuzi_email').rename(columns=partner_col)

In [41]:
result_export = result.copy()
payment_cols = [c for c in result_export.columns if c.endswith('(Payment)')]
count_cols = [c for c in result_export.columns if c.endswith('(Count)')]
result_export[payment_cols] = result_export[payment_cols].round(2)
result_export[count_cols] = result_export[count_cols].astype(int)

with pd.ExcelWriter(
    PATHS['report_out'],
    mode='a',
    if_sheet_exists='replace',
    engine='openpyxl',
) as writer:
    learner_demographics.to_excel(writer, index=False, sheet_name='Learner Demographics')
    singles.to_excel(writer, index=False, sheet_name='One Per Learner')
    wide.to_excel(writer, index=False, sheet_name='Monthly Entries Breakdown')
    result_export.to_excel(writer, index=False, sheet_name='Summarized Opportunities')
    breakdown_export.to_excel(writer, index=False, sheet_name='Earning Opportunities Secured')
